# Tệp: notebooks/moocradar_colab.ipynb
# Mục đích: Tái lập bước chuẩn hóa, huấn luyện, đánh giá và lộ trình học trên Google Colab.

Đặt repository và bốn file raw trên Google Drive. Kiểm tra các đường dẫn bên dưới rồi chạy cell theo thứ tự. `smoke_test` dùng 25 sinh viên để kiểm tra nhanh; `full_run` dùng toàn bộ tương tác cục bộ. Artifacts và checkpoint JSON được lưu trực tiếp trên Drive. Nên chạy notebook [Data Quality & EDA](01_data_quality_eda.ipynb) trước bước huấn luyện.


In [16]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
# Clone mã nguồn dự án MOOC-Radar từ GitHub và đặt tên thư mục trùng khớp với PROJECT_ROOT
!git clone https://github.com/DuoChip/Personalized-Learning-Path /content/Personalized-Learning-Path

fatal: destination path '/content/Personalized-Learning-Path' already exists and is not an empty directory.


In [18]:
from pathlib import Path
import os
import subprocess

PROJECT_ROOT = Path("/content/Personalized-Learning-Path")
DATA_ROOT = Path("/content/drive/MyDrive/MOOC-Radar/data/raw")
PROCESSED_ROOT = Path("/content/drive/MyDrive/MOOC-Radar/data/processed")
ARTIFACT_ROOT = Path("/content/drive/MyDrive/MOOC-Radar/artifacts")
MODE = "full_run"  # Change to "full_run" after reviewing the smoke outputs.
assert (PROJECT_ROOT / "requirements.txt").exists(), "Set PROJECT_ROOT to the repository folder."
assert (DATA_ROOT / "problem.json").exists(), "Place the raw files in DATA_ROOT."
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


In [19]:
subprocess.run(["python", "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)


CompletedProcess(args=['python', '-m', 'pip', 'install', '-r', '/content/Personalized-Learning-Path/requirements.txt'], returncode=0)

In [20]:
import shutil
print("Số lõi CPU:", os.cpu_count())
print("RAM (GiB):", round(os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / 2**30, 2))
print("Dung lượng trống (GiB):", round(shutil.disk_usage("/content").free / 2**30, 2))
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có; pipeline hỗ trợ CPU")
except ImportError:
    print("GPU check không có; pipeline hỗ trợ CPU")


Số lõi CPU: 2
RAM (GiB): 12.67
Dung lượng trống (GiB): 87.04
GPU: không có; pipeline hỗ trợ CPU


## Chuẩn bị dữ liệu

Bước này đọc JSON array theo từng chuỗi, kiểm tra chất lượng và lưu các bảng chuẩn hóa. Run mẫu vẫn đọc đầy đủ problem metadata. Dữ liệu gốc không bị chỉnh sửa.


In [21]:
processed_dir = PROCESSED_ROOT / ("smoke" if MODE == "smoke_test" else "full")
env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT / "src"), "MPLCONFIGDIR": "/content/.mplconfig"}
prepare_cmd = ["python", "-m", "moocradar", "prepare", "--data-root", str(DATA_ROOT), "--output-dir", str(processed_dir)]
if MODE == "smoke_test":
    prepare_cmd += ["--max-students", "25"]
subprocess.run(prepare_cmd, cwd=PROJECT_ROOT, env=env, check=True)


CompletedProcess(args=['python', '-m', 'moocradar', 'prepare', '--data-root', '/content/drive/MyDrive/MOOC-Radar/data/raw', '--output-dir', '/content/drive/MyDrive/MOOC-Radar/data/processed/full'], returncode=0)

## Huấn luyện, đánh giá và xuất hồ sơ/lộ trình

Huấn luyện tạo thư mục run mới. Cùng một lệnh sẽ xuất metrics validation/test và các bảng cho frontend. Tham số BKT được lưu dưới dạng JSON để suy luận mà không cần train lại.


In [22]:
before = set(ARTIFACT_ROOT.glob("*/manifest.json"))
subprocess.run(["python", "-m", "moocradar", "train", "--processed-dir", str(processed_dir), "--artifact-root", str(ARTIFACT_ROOT), "--seed", "42"], cwd=PROJECT_ROOT, env=env, check=True)
new_manifests = set(ARTIFACT_ROOT.glob("*/manifest.json")) - before
assert len(new_manifests) == 1
run_dir = next(iter(new_manifests)).parent
print("Run đã lưu:", run_dir)


Run đã lưu: /content/drive/MyDrive/MOOC-Radar/artifacts/20261017T160339717197Z_b38447


In [23]:
import csv
import json
manifest = json.loads((run_dir / "manifest.json").read_text())
assert manifest["status"] == "complete"
with (run_dir / "metrics.csv").open() as f:
    for row in csv.DictReader(f):
        print(row)
for name in ("data_quality.json", "predictions.csv", "student_mastery.csv", "cognitive_profile.csv", "knowledge_gaps.csv", "learning_paths.csv", "models/bkt.json"):
    assert (run_dir / name).exists(), name
print("Hồ sơ và lộ trình:", run_dir / "student_mastery.csv", run_dir / "learning_paths.csv")


{'model': 'bkt', 'split': 'test', 'n': '145307', 'roc_auc': '0.653821', 'accuracy': '0.80332', 'f1': '0.890934', 'brier': '0.148772', 'tp': '116728', 'tn': '0', 'fp': '28579', 'fn': '0'}
{'model': 'bkt', 'split': 'validation', 'n': '128789', 'roc_auc': '0.663296', 'accuracy': '0.820435', 'f1': '0.901361', 'brier': '0.138928', 'tp': '105663', 'tn': '0', 'fp': '23126', 'fn': '0'}
{'model': 'smoothed_concept', 'split': 'test', 'n': '145307', 'roc_auc': '0.738546', 'accuracy': '0.811255', 'f1': '0.894076', 'brier': '0.138044', 'tp': '115748', 'tn': '2133', 'fp': '26446', 'fn': '980'}
{'model': 'smoothed_concept', 'split': 'validation', 'n': '128789', 'roc_auc': '0.735351', 'accuracy': '0.827276', 'f1': '0.903734', 'brier': '0.129782', 'tp': '104417', 'tn': '2127', 'fp': '20999', 'fn': '1246'}
Hồ sơ và lộ trình: /content/drive/MyDrive/MOOC-Radar/artifacts/20261017T160339717197Z_b38447/student_mastery.csv /content/drive/MyDrive/MOOC-Radar/artifacts/20261017T160339717197Z_b38447/learning_path

## Demo JSON và chia sẻ kết quả

Chọn một sinh viên từ bảng tương tác đã lưu. Có thể thêm một bài và nhãn mới để cập nhật trạng thái mà không cần train lại. Tải cả thư mục run hoàn chỉnh trên Drive để bàn giao cho nhóm dashboard.


In [24]:
with (processed_dir / "interactions.csv").open() as f:
    user_id = next(csv.DictReader(f))["user_id"]
subprocess.run(["python", "-m", "moocradar", "demo", "--run-dir", str(run_dir), "--processed-dir", str(processed_dir), "--user-id", user_id], cwd=PROJECT_ROOT, env=env, check=True)
print("Chia sẻ thư mục Drive này:", run_dir)


Chia sẻ thư mục Drive này: /content/drive/MyDrive/MOOC-Radar/artifacts/20261017T160339717197Z_b38447
